In [1]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "Artificial is intelligence is transforming the way people the interact with technology"
tokens = u_tokenizer.encode(prompt)

tokens

tensor([ 1, 63,  3, 63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  5, 63,
         8, 63,  9, 63, 10])

In [2]:
context_length = 32

In [3]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab),embedding_dim = 4,num_heads = 2,context_length= 32,num_layers = 3)

out = u_model(tokens)
out.shape

torch.Size([23, 65])

In [4]:
out = u_model.generate(tokens,3)
u_tokenizer.decode(out)

'artificial is intelligence is transforming the way people the interact with technologymodelswaymodels'

In [5]:
with open("text.txt", "r") as f:
    text = f.read()

len(text) ,text[:100]

(1970,
 'Artificial intelligence is transforming the way people interact with technology. Large language mode')

In [6]:
token_ids = u_tokenizer.encode(text)
len(token_ids) , type(token_ids)

(1000, torch.Tensor)

In [7]:
ids = token_ids.detach().cpu().numpy().tolist()
len(ids) ,type(ids)

(1000, list)

In [8]:
from text_dataset import TextDataset

stride = 2

dataset = TextDataset(ids , context_length,stride)

len(dataset.inputs) , len(dataset.targets)

(484, 484)

In [9]:
dataset.inputs[0],dataset.targets[0]

(tensor([ 1, 63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  8, 63,  9, 63,
         10,  0, 63, 11, 63, 12, 63, 13, 63, 14, 63, 15, 63, 16]),
 tensor([63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  8, 63,  9, 63, 10,
          0, 63, 11, 63, 12, 63, 13, 63, 14, 63, 15, 63, 16, 63]))

In [10]:
#model parameters count 
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

#model architecture 
print(u_model)

1089
UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(65, 4)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=4, out_features=4, bias=True)
        (up_proj): Linear(in_features=4, out_features=4, bias=True)
        (down_proj): Linear(in_features=4, out_features=4, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projectio

In [11]:
out0 = u_model(dataset.inputs[0])
out0.shape

torch.Size([32, 65])

In [12]:
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()

In [13]:
loss = loss_fn(out0,dataset.targets[0])
loss

tensor(4.0208, grad_fn=<NllLossBackward0>)

In [14]:
#optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(),lr = 1e-3)

In [15]:
for input , target in dataset :
    print(input.shape , target.shape)
    break 

torch.Size([32]) torch.Size([32])


In [16]:
epoch = 1000000
for epoch in range(epoch):
    total_loss = 0
    for input, target in dataset :
        pred = u_model(input)
        loss = loss_fn(pred,target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    average_loss = total_loss/ len(dataset)
    print(f"Epoch {epoch} loss : {loss.item()} average loss : {average_loss}")

Epoch 0 loss : 1.252316951751709 average loss : 2.0408234705856025
Epoch 1 loss : 1.2102411985397339 average loss : 1.7635106541647398
Epoch 2 loss : 1.2009930610656738 average loss : 1.7509056039092954
Epoch 3 loss : 1.1392942667007446 average loss : 1.7278111115221149
Epoch 4 loss : 1.1883498430252075 average loss : 1.6676499567741205
Epoch 5 loss : 1.1924388408660889 average loss : 1.5900100661703378
Epoch 6 loss : 1.1902278661727905 average loss : 1.5483151134873225
Epoch 7 loss : 1.2518579959869385 average loss : 1.5132550589801852
Epoch 8 loss : 1.1634684801101685 average loss : 1.4891019914268462
Epoch 9 loss : 1.1626054048538208 average loss : 1.4645014839970376
Epoch 10 loss : 1.149387001991272 average loss : 1.4454285993802647
Epoch 11 loss : 1.1215462684631348 average loss : 1.4284918593966272
Epoch 12 loss : 1.1709469556808472 average loss : 1.4149101465201575
Epoch 13 loss : 1.1599011421203613 average loss : 1.401994988568558
Epoch 14 loss : 1.143403172492981 average loss 

KeyboardInterrupt: 

In [17]:
new_tokens = tokens.detach().cpu().numpy().tolist()
new_tokens.pop()
new_tokens.pop()
new_tokens

[1, 63, 3, 63, 2, 63, 3, 63, 4, 63, 5, 63, 6, 63, 7, 63, 5, 63, 8, 63, 9]

In [18]:
import torch 
new_tokens = u_tokenizer.encode(prompt)
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(63)

out = u_model(torch.tensor(new_tokens))
probs = torch.softmax(out[-1] , dim =-1)
max_prob ,max_index = torch.max(probs,dim =-1)
max_prob , max_index , probs

(tensor(0.4175, grad_fn=<MaxBackward0>),
 tensor(0),
 tensor([4.1748e-01, 1.0045e-03, 1.6140e-03, 3.6306e-03, 1.5634e-03, 5.1804e-02,
         2.7116e-03, 1.8693e-03, 2.0502e-03, 4.7310e-03, 2.6119e-03, 2.1149e-03,
         1.3054e-02, 1.0193e-02, 2.6861e-03, 4.6292e-03, 5.3912e-03, 5.3261e-03,
         8.6261e-03, 2.3681e-03, 3.0701e-03, 3.3250e-03, 3.2699e-03, 1.0154e-02,
         2.6582e-03, 2.0697e-03, 7.1870e-03, 3.1864e-03, 2.6140e-03, 2.3869e-03,
         6.1302e-03, 6.1908e-03, 2.1314e-03, 2.7084e-03, 2.6409e-03, 2.2035e-03,
         2.3751e-03, 2.2972e-03, 2.2593e-03, 2.6155e-03, 2.7235e-03, 2.0649e-03,
         2.5635e-03, 2.0097e-03, 2.0817e-03, 2.3298e-03, 1.0067e-02, 2.2208e-03,
         1.1303e-02, 1.6104e-03, 1.5969e-03, 1.9340e-03, 1.6302e-03, 1.5646e-03,
         4.3907e-03, 2.2937e-03, 4.8748e-03, 2.2261e-03, 1.8201e-03, 2.3104e-03,
         2.8965e-03, 2.6064e-03, 3.4308e-03, 3.1051e-01, 1.4090e-05],
        grad_fn=<SoftmaxBackward0>))

In [17]:
#save model 
torch.save(u_model.state_dict() ,"u_model.pth")

#load model
u_model.load_state_dict(torch.load("u_model.pth"))

#generate text
new_tokens = u_tokenizer.encode(prompt)
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(63)
len(new_tokens)


24

In [19]:
out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1],dim =-1)
max_prob , max_index = torch.max(probs,dim=-1)
max_prob , max_index , probs

(tensor(0.0624, grad_fn=<MaxBackward0>),
 tensor(0),
 tensor([6.2415e-02, 1.1079e-03, 9.4700e-03, 5.6578e-03, 9.8742e-03, 2.9045e-02,
         1.1182e-02, 1.1486e-02, 1.0991e-02, 1.7097e-02, 8.5406e-03, 7.7526e-03,
         5.1653e-02, 1.3677e-02, 1.3060e-02, 2.0316e-02, 4.3507e-02, 4.8156e-02,
         1.8155e-02, 1.7320e-02, 1.7612e-02, 1.6090e-02, 1.5483e-02, 1.0703e-02,
         9.7586e-03, 8.8627e-03, 7.3230e-03, 7.4512e-03, 6.9730e-03, 4.7216e-03,
         4.9565e-03, 1.9583e-03, 2.0590e-03, 1.8859e-03, 2.9452e-03, 1.2951e-02,
         1.3785e-02, 1.2802e-02, 9.7102e-03, 1.6701e-02, 1.8550e-02, 2.9266e-02,
         2.8518e-02, 2.6333e-02, 2.4835e-02, 2.4111e-02, 2.1831e-02, 2.3481e-02,
         2.4504e-02, 2.6387e-02, 2.2218e-02, 2.3643e-02, 2.2717e-02, 2.0566e-02,
         1.4856e-02, 1.3868e-02, 1.2001e-02, 7.2562e-03, 2.7728e-03, 7.3025e-04,
         7.2342e-04, 4.9485e-04, 2.0066e-04, 1.4941e-02, 8.4913e-09],
        grad_fn=<SoftmaxBackward0>))

In [20]:
loaded_model = UstaModel(65 , embedding_dim=4 , num_heads = 2, context_length = 32 ,num_layers = 3)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model

UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(65, 4)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=4, out_features=4, bias=True)
        (up_proj): Linear(in_features=4, out_features=4, bias=True)
        (down_proj): Linear(in_features=4, out_features=4, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): L

In [21]:
import torch 
new_tokens = u_tokenizer.encode(prompt)
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(63)
u_model.generate(torch.tensor(new_tokens),2)

[1,
 63,
 3,
 63,
 2,
 63,
 3,
 63,
 4,
 63,
 5,
 63,
 6,
 63,
 7,
 63,
 5,
 63,
 8,
 63,
 9,
 63,
 10,
 63,
 0,
 0]